# GEB parameter sensitivity

Evaluate GEB over the specified values of $m$ and $\alpha$ on Dry Bean and the two-dimensional Gaussian mixture. Both datasets use the Gaussian transform. Completed fits in the result CSV are skipped on rerun.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'experiment_utils' / 'run_benchmark.py').exists():
    raise FileNotFoundError('Open this notebook from the project root or notebooks folder.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiment_utils.transforms import apply_normal_cdf, fit_normal_cdf
import experiment_utils.run_benchmark as benchmark
fit_and_score_geb = benchmark.fit_and_score_geb

## Configuration

In [ ]:
DATASETS = {
    'gaussian_mixture_low_d': {'label': 'Gaussian mixture ($D=2$)', 'transform': 'normal'},
    'dry_bean': {'label': 'Dry Bean', 'transform': 'normal'},
}
REPETITIONS = range(1, 21)
ALPHAS = [0.01, 0.1, 1, 10, 100, 1000, 10000]
GRID_INTERVALS = [2, 4, 8, 16, 32, 64, 128, 256, 512]
FIXED_M = 32
FIXED_ALPHA = 1.0
OVERWRITE = False

OUTPUT_DIR = PROJECT_ROOT / 'results' / 'parameter_sensitivity'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FILE = OUTPUT_DIR / 'geb_parameter_sensitivity.csv'

pd.DataFrame([
    {'dataset': key, 'display_name': value['label'], 'transformation': value['transform']}
    for key, value in DATASETS.items()
])

## Evaluation and result saving


In [ ]:
COLUMNS = [
    'dataset', 'display_name', 'transformation', 'repetition',
    'experiment', 'setting', 'test_log_score', 'log_marginal_likelihood',
]

def atomic_write_csv(table, path):
    temporary = path.with_suffix(path.suffix + '.tmp')
    table.to_csv(temporary, index=False)
    temporary.replace(path)

def evaluate(dataset, transformation, repetition, m, alpha):
    x_train = benchmark.load_split(dataset, repetition, 'train')
    x_test = benchmark.load_split(dataset, repetition, 'test')
    parameters = fit_normal_cdf(x_train)
    train, train_log_base = apply_normal_cdf(x_train, parameters, jacobian=True)
    test, test_log_base = apply_normal_cdf(x_test, parameters, jacobian=True)
    test_log_density, _, tree = fit_and_score_geb(
        train, test, m=m, alpha=alpha
    )
    test_score = float((test_log_density + test_log_base).mean())
    tree.init_log_marginal_likelihood(tree.root)
    log_marginal = (
        tree.root.root_log_marginal_likelihood + train_log_base.sum()
    ) / len(train)
    return test_score, float(log_marginal)

def result_key(row):
    return (row['dataset'], row['experiment'], float(row['setting']), int(row['repetition']))

if RESULTS_FILE.exists() and not OVERWRITE:
    results = pd.read_csv(RESULTS_FILE)
    if list(results.columns) != COLUMNS:
        raise ValueError('Existing sensitivity checkpoint has incompatible columns.')
else:
    results = pd.DataFrame(columns=COLUMNS)

completed = {result_key(row) for row in results.to_dict('records')}
print(f'Loaded {len(results)} completed fits from {RESULTS_FILE}')

## Run the two sensitivity experiments

In [ ]:
total_expected = len(DATASETS) * (len(ALPHAS) + len(GRID_INTERVALS)) * len(REPETITIONS)
for dataset, specification in DATASETS.items():
    label = specification['label']
    transformation = specification['transform']
    experiments = (
        [('alpha', alpha, FIXED_M, alpha) for alpha in ALPHAS]
        + [('M', m, m, FIXED_ALPHA) for m in GRID_INTERVALS]
    )
    for experiment, setting, m, alpha in experiments:
        for repetition in REPETITIONS:
            key = (dataset, experiment, float(setting), repetition)
            if key in completed:
                continue
            score, marginal = evaluate(dataset, transformation, repetition, m, alpha)
            row = {
                'dataset': dataset, 'display_name': label,
                'transformation': transformation, 'repetition': repetition,
                'experiment': experiment, 'setting': setting,
                'test_log_score': score,
                'log_marginal_likelihood': marginal,
            }
            results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)
            results = results.sort_values(
                ['dataset', 'experiment', 'setting', 'repetition']
            ).reset_index(drop=True)
            atomic_write_csv(results, RESULTS_FILE)
            completed.add(key)
            print(
                f'{len(results):4d}/{total_expected}: {label}, '
                f'{experiment}={setting:g}, repetition={repetition:02d}, '
                f'score={score:.6f}', flush=True,
            )

print(f'Completed {len(results)}/{total_expected} fits.')

## Summaries

In [ ]:
long_results = results.rename(columns={
    'test_log_score': 'Test log score',
    'log_marginal_likelihood': 'Average log marginal likelihood',
}).melt(
    id_vars=['dataset', 'display_name', 'transformation',
             'repetition', 'experiment', 'setting'],
    var_name='metric', value_name='value',
)
summary = long_results.groupby(
    ['dataset', 'display_name', 'transformation',
     'experiment', 'setting', 'metric']
).value.agg(
    mean='mean', q25=lambda x: x.quantile(.25),
    q75=lambda x: x.quantile(.75), repetitions='count',
).reset_index()
summary_file = OUTPUT_DIR / 'geb_parameter_sensitivity_summary.csv'
summary.to_csv(summary_file, index=False)
summary

## Figures


In [ ]:
COLORS = {'Test log score': '#4C78A8', 'Average log marginal likelihood': '#D95F5F'}

def plot_dataset(dataset, specification):
    table = summary[summary.dataset == dataset]
    if table.empty:
        print(f'Skipping {dataset}: no completed results.')
        return
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, experiment, xlabel in [
        (axes[0], 'alpha', rf'$\alpha$ ($m={FIXED_M}$)'),
        (axes[1], 'M', rf'$m$ ($\alpha={FIXED_ALPHA:g}$)'),
    ]:
        experiment_table = table[table.experiment == experiment]
        for metric in ['Test log score', 'Average log marginal likelihood']:
            values = experiment_table[experiment_table.metric == metric]
            x = values.setting.to_numpy(dtype=float)
            mean = values['mean'].to_numpy(dtype=float)
            q25 = values.q25.to_numpy(dtype=float)
            q75 = values.q75.to_numpy(dtype=float)
            ax.plot(x, mean, marker='o', linewidth=2.0, markersize=5, color=COLORS[metric], label=metric)
            ax.fill_between(x, q25, q75, color=COLORS[metric], alpha=.15)
        ax.set_xscale('log', base=10 if experiment == 'alpha' else 2)
        ax.set_xlabel(xlabel, fontsize=12)
        ax.set_ylabel('Per-observation log score', fontsize=12)
        ax.tick_params(labelsize=10)
    axes[0].legend(frameon=False, fontsize=10)
    fig.suptitle(specification['label'], fontsize=13)
    fig.tight_layout()
    figure_file = OUTPUT_DIR / f'geb_sensitivity_{dataset}.pdf'
    fig.savefig(figure_file, bbox_inches='tight')
    display(fig)
    plt.close(fig)
    print('Saved', figure_file)

for dataset, specification in DATASETS.items():
    plot_dataset(dataset, specification)